In [1]:
import json
import os
from pathlib import Path
import imageio_ffmpeg
import whisper
root = Path.home() / "Desktop/AlgoTrading"
audio_path = root / "data/audio/KCNKpyP-DNQ.mp3"

# Whisper вызывает FFmpeg по имени.
ffmpeg_dir = root / ".venv/bin"
ffmpeg_path = ffmpeg_dir / "ffmpeg"

if not ffmpeg_path.exists():
    ffmpeg_path.symlink_to(imageio_ffmpeg.get_ffmpeg_exe())

os.environ["PATH"] = (
    str(ffmpeg_dir) + os.pathsep + os.environ.get("PATH", "")
)

if not audio_path.is_file():
    raise FileNotFoundError(audio_path)

model = whisper.load_model(
    "small",
    download_root=str(root / "Models/whisper"),
)

result = model.transcribe(
    str(audio_path),
    language="ru",
    task="transcribe",
    fp16=False,
)

save_dir = root / "data/transcripts"
save_dir.mkdir(parents=True, exist_ok=True)

(save_dir / f"{audio_path.stem}.txt").write_text(
    result["text"],
    encoding="utf-8",
)

# Текст с таймкодами для будущего RAG.
transcript = {
    "video_id": audio_path.stem,
    "source_url": f"https://www.youtube.com/watch?v={audio_path.stem}",
    "language": result["language"],
    "segments": [
        {
            "start": segment["start"],
            "end": segment["end"],
            "text": segment["text"].strip(),
        }
        for segment in result["segments"]
    ],
}

(save_dir / f"{audio_path.stem}.json").write_text(
    json.dumps(transcript, ensure_ascii=False, indent=2),
    encoding="utf-8",
)



100%|███████████████████████████████████████| 461M/461M [01:22<00:00, 5.87MiB/s]


66028